# The 'So What?' Test

**DS4DH · Module 11 — Policy and Domain Reasoning**

*Technique:* Classifying findings by what they license — action, monitoring, or context only

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/11a_evidence_classification.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Statistical significance is a low bar for policy. A finding earns a recommendation
only if it clears three hurdles in order:

1. **Real** — survives correction for multiple testing, with a usable effect size
2. **Modifiable** — points at something an institution can actually change
3. **Owned** — some institution's remit covers that thing

Fail hurdle 1 and you have noise. Pass 1 but fail 2 and you have context. Pass 1
and 2 but fail 3 and you have a finding with nowhere to send it.

This notebook runs every finding from the course through the three hurdles.

In [ ]:
csd = df.dropna(subset=['csd_code'])

imm = csd[csd['immigrant_status'] == 'Immigrant'][
    ['csd_code', 'geography_name', 'cma', 'Renter']].copy()
imm.columns = ['csd_code', 'geography_name', 'cma', 'renter_stir_imm']

nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']].copy()
nim.columns = ['csd_code', 'renter_stir_nim']

penalty_df = imm.merge(nim, on='csd_code', how='inner')
penalty_df['penalty'] = penalty_df['renter_stir_imm'] - penalty_df['renter_stir_nim']
penalty_df = penalty_df.dropna(subset=['penalty'])
penalty_df = penalty_df[penalty_df['cma'].isin(CITIES)]

print(f'{len(penalty_df)} CSDs where BOTH groups have a reported renter STIR')
print()
print(penalty_df[['geography_name', 'cma', 'renter_stir_imm',
                  'renter_stir_nim', 'penalty']].head(8).to_string(index=False))

In [ ]:
# Hurdle 1, applied to the course's five headline findings.
ALPHA = 0.05
BONF = ALPHA / len(CITIES)

findings = []
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    t, p = stats.ttest_1samp(s, 0)
    d = s.mean() / s.std(ddof=1)
    findings.append({
        'finding': f'{city}: immigrant renter gap',
        'n': len(s), 'effect': s.mean(), 'p': p, 'd': d,
        'survives': bool(p < BONF),
    })

fdf = pd.DataFrame(findings)
print(fdf.round(4).to_string(index=False))
print()
print(f'{fdf["survives"].sum()} of {len(fdf)} findings survive Bonferroni at α={BONF}.')

## Hurdle 2 — is it modifiable?

This one cannot be computed. It is a question about the world, and the honest
answer for most descriptive findings is no.

A factor is modifiable if some intervention could plausibly change it within a
policy time horizon. "Immigrant status" is not modifiable. "Rent supplement
eligibility" is. The distinction determines whether a finding can become a
recommendation or can only ever be a targeting criterion.

In [ ]:
hurdle2 = {
    'immigrant status':        (False, 'a characteristic of households, not a lever'),
    'household income':        (True,  'income supplements, wage policy, transfers'),
    'shelter cost':            (True,  'rent regulation, supply, subsidy'),
    'which city you live in':  (False, 'not a policy instrument in any useful sense'),
    'municipality population': (False, 'proxy for market depth; not directly actionable'),
}

print(f'{"factor":<26}{"modifiable":>12}   why')
print('-' * 78)
for k, (mod, why) in hurdle2.items():
    print(f'{k:<26}{("yes" if mod else "no"):>12}   {why}')
print()
print('The Edmonton finding is about immigrant status — a targeting variable.')
print('It tells you WHO to look at, never WHAT to change.')

In [ ]:
# Hurdle 3 — who owns it? Same finding, four different readers.
readers = {
    'municipal planning':  'zoning, supply, development approvals',
    'CMHC (federal)':      'mortgage insurance, national housing strategy funding',
    'tenant advocacy':     'rights enforcement, targeted outreach, casework',
    'OSFI (regulator)':    'lender risk exposure, not household affordability',
}
for who, remit in readers.items():
    print(f'{who:<22}{remit}')
print()
print('A finding about immigrant renter burden in 13 Edmonton subdivisions is')
print('actionable for tenant advocacy outreach, marginal for municipal planning,')
print('and outside OSFI\'s remit entirely. Same number, three different verdicts.')

## The classification

Three categories, and only one of them supports a recommendation.

In [ ]:
def classify(row):
    if not row['survives']:
        return 'context only — not distinguishable from noise'
    if abs(row['d']) < 0.2:
        return 'monitor — real but too small to act on'
    return 'actionable — subject to the modifiability test'

fdf['class'] = fdf.apply(classify, axis=1)
print(fdf[['finding', 'n', 'effect', 'p', 'd', 'class']].round(3).to_string(index=False))
print()
print('Note that "actionable" here still means "actionable as a targeting')
print('criterion" — hurdle 2 caps what the Edmonton result can support.')

### 🔧 Your turn 1

Add a fifth finding to `fdf`: the pooled result across all four cities
(`stats.ttest_1samp(penalty_df['penalty'], 0)`).

Does pooling change the classification? Is pooling across cities a legitimate way
to rescue a finding that failed city by city, or is it a fourth test?

### 🔧 Your turn 2

Take the Edmonton result and write the two sentences you would send to (a) a
tenant advocacy organisation and (b) a municipal planning department.

They should differ, and neither should overstate what a sample of 13 subdivisions
supports.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Pooling gives a small, non-significant overall effect, so it does
not rescue anything. More importantly, running it *after* seeing the city-level
results is a fifth test on the same data and should be counted in the correction.
Pooling is legitimate when specified in advance as the primary analysis; it is not
legitimate as a fallback when the pre-specified tests disappointed. The
distinction is about when you decided, which is invisible in the output and
visible only in your own honesty.

**Your turn 2.** For tenant advocacy:

> In Edmonton's census subdivisions with comparable data, immigrant renter
> households spend on average 3.2 percentage points less of their income on
> shelter than non-immigrant renters in the same municipalities. This is the only
> immigrant/non-immigrant difference in the four-city dataset that survives
> correction for multiple comparisons, and it runs opposite to the direction
> usually assumed — outreach materials should not assume immigrant renters are
> the more burdened group in this market.

For municipal planning:

> We find no evidence of an immigrant-specific renter burden gap in Edmonton;
> if anything the gap runs the other way. Housing burden here varies far more by
> municipality than by immigrant status, so planning responses should be
> geographic rather than group-targeted.

Both are true; they differ because the two readers can act on different things.

</details>

## Where this stops

You can now sort findings into what they license. Turning the one surviving
finding into a numeric recommendation is the next notebook.